# AUDIT-X v4 · 15 · Second-round reviewer analyses (CPU only)

Uses the cached outputs of notebooks **01–03** (training seed 0) and **13a–13c** (training seeds 1–2), all under protocol v4.

| Section | Reviewer point |
|---|---|
| **S1** | Configuration selection without unknown families: inflation measured against the pool mean, its dependence on pool size, and a known-data-only selection rule |
| **S2** | Family-level cluster bootstrap over splits and training seeds, per-dataset effect sizes with 95 % intervals, two-sided tests, Holm correction; single models versus the deep ensemble |
| **S3** | Low false-alarm operating points: finite-sample conformal thresholds, Clopper–Pearson intervals for FAR, bootstrap intervals for ZDR, repeated calibration draws, reliability flags |
| **S4** | Do the rankings change at a better KNN neighbourhood (k = 5, 10) than the fixed k = 50? |
| **S5** | AGEF-Net per-seed results and training curves (is it under-trained?) |
| **S6** | Early-exit compute for known and unknown inputs separately |

**Settings fixed before running** (cell 1): 500 bootstrap draws; Holm correction at α = 0.05 within each family of comparisons;
a 1 % operating point is called *reliable* only if the calibration set has at least 10 expected exceedances (n·α ≥ 10);
200 repeated calibration draws (calibration and test known images pooled and re-split 50/50).

**Inputs:** committed outputs of notebooks **01, 02, 03, 13a, 13b, 13c** (protocol v4). Accelerator **None**. ≈ 1.5–2.5 h (estimate).

In [ ]:
# Cell 1 - settings (fixed before running)
N_BOOT = 500
HOLM_ALPHA = 0.05
FAR_LOW = 0.01
MIN_EXCEEDANCES = 10
N_REDRAWS = 200
KNN_KS = [5, 10, 50]
SCORES = ["msp", "entropy", "maxlogit", "energy", "knn5", "knn10", "knn50", "mahalanobis"]

In [ ]:
%%writefile /kaggle/working/auditx_eval_v2.py
# =============================================================================
# AUDIT-X  —  corrected evaluation pipeline (v2)
# =============================================================================
# One script that produces every number the paper needs, from saved artefacts.
#
# What it fixes relative to the original notebooks
#   1. Known data are split per run into  train / val / calib / test  (70/10/10/10).
#      - val   : epoch selection only (known classes only)
#      - calib : operating thresholds only (target FAR)
#      - test  : FAR and closed-set accuracy are REPORTED here, so FAR is a
#                measurement, not a tautology. Training samples are never scored.
#   2. Repeated over several random unknown-class splits (SPLIT_SEEDS); results are
#      mean ± std across splits, with bootstrap CIs and paired tests across splits.
#   3. Scores are computed in float64 from logits; non-saturating scores
#      (MaxLogit, Energy) are reported next to MSP / entropy, plus the fraction of
#      calibration scores tied at the threshold (saturation diagnostic).
#   4. Threshold-free metrics (AUROC, AUPR-out, FPR@95TPR) are primary;
#      ZDR at a calibrated 5% FAR is the operational metric.
#   5. Real baselines: single CNNs, a jointly-trained average ensemble, an
#      independently-trained deep ensemble, a genuine OpenMax (Weibull on distances
#      to class MAVs), Energy, MaxLogit, KNN on penultimate features.
#   6. Four-tag triage is reported with per-tag fractions on test-known and unknown.
#   7. Optional dynamic early exit that is actually conditional: the exit rule uses
#      MobileNet's own confidence only (the gate needs all three experts, so a
#      gate-entropy exit rule can never save compute). Cost is measured with
#      torch.utils.flop_counter and wall-clock timing per expert.
#   8. Classes named in NEVER_UNKNOWN (benign / "Other") are never held out as
#      zero-day, so flagging benign files cannot count as zero-day recall.
#   9. Everything is resumable: per-run scores are saved to .npz and skipped if present.
#
# Adversarial, federated, concept-drift and SOC-stream experiments are NOT here on
# purpose; see the accompanying notes for why they need redesign first.
#
# Usage (Kaggle cell):   !python auditx_eval_v2.py
# Usage (subset):        !python auditx_eval_v2.py --datasets Malimg --seeds 0 1
# Smoke test:            python auditx_eval_v2.py --smoke
# =============================================================================
import argparse, copy, json, math, os, time, warnings, zlib
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms
from scipy import stats
from scipy.special import logsumexp, softmax
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning)

# -----------------------------------------------------------------------------
# 0. Configuration (edit paths here, or override from the command line)
# -----------------------------------------------------------------------------
CFG = dict(
    DATASETS=OrderedDict(
        Malimg="/kaggle/input/datasets/mohammadmehedimunna/malimg-dataset-25/Malimg_Dataset_25classes",
        MaleVis="/kaggle/input/datasets/mohammadmehedimunna/melevis/MaleVis",
        Dumpware10="/kaggle/working/dumpware10_merged",   # built from dumpware10_images/224/224/{TRAINING,TEST}
    ),
    # matched case-insensitively against class folder names; these are never held out as zero-day
    NEVER_UNKNOWN=["other", "others", "benign", "goodware", "clean", "legitimate"],
    UNKNOWN_RATIO=0.2,
    SPLIT_SEEDS=[0, 1, 2, 3, 4],
    FRACS=(0.70, 0.10, 0.10, 0.10),          # train / val / calib / test (known classes)
    EPOCHS=12,
    BATCH=32,
    LR=1e-3,
    WD=1e-4,
    IMG=224,
    WORKERS=4,
    PRETRAINED=True,
    AMP=True,
    TARGET_FAR=0.05,
    # Models to train per split. Keys are defined in MODEL_SPECS below.
    MODELS=["agef_sharp", "avg_joint", "single_mobilenet", "single_resnet18", "single_effnet_b0"],
    DEEP_ENSEMBLE=True,                        # built post hoc from the three single_* models
    TRIAGE_AXES=("msp", "entropy"),            # the paper's two axes
    KNN_K=50,
    KNN_MAX_TRAIN=6000,
    OPENMAX_TAIL=20,
    OPENMAX_ALPHA=10,
    RUN_DYNAMIC=True,
    DYNAMIC_EXIT_QUANTILES=[0.0, 0.25, 0.5, 0.75, 0.9],
    N_BOOT=10000,
    OUT_DIR="/kaggle/working/auditx_v2",
    # Duplicate-aware splitting (protocol v3): byte-identical or pixel-identical images form one group; a group is never
    # split across train/val/calib/test; groups containing more than one family (label conflicts) are removed; classes that
    # cannot be split (one group > 90 % of the class, or < 10 images left) are excluded from the experiment.
    # Near-identical images (perceptual hashes) are NOT grouped: low-information byte plots share identical dHashes and
    # transitive matching merges whole families; they are reported as a sensitivity analysis (notebook 12b).
    GROUPING=True,
    GROUP_KEYS=(0, 1),            # 0 = file MD5, 1 = pixel MD5, 2 = 256-bit dHash
    UNSPLITTABLE_SHARE=0.9,
    MIN_CLASS_IMAGES=10,
    # Near-duplicate grouping (protocol v4, applied to the datasets listed): within each family, images whose 256-bit dHash
    # differs by <= NEAR_DUP_BITS are merged, closest pairs first, and a merge is refused if the group would exceed
    # NEAR_DUP_CAP images, which prevents transitive chaining across whole families.
    # cap 250 = smallest cap that brings residual near-identical matches (256-bit dHash within 8 bits) between training and
    # evaluation images below 5 % on every dataset without making a class unsplittable through chaining; chosen from duplicate
    # statistics only, never from detection results.
    NEAR_DUP_DATASETS=("Malimg", "MaleVis", "Dumpware10"),
    NEAR_DUP_BITS=8,
    NEAR_DUP_CAP=250,
)
PROTOCOL = "grouped-L1-v3"

EXPERT_NAMES = ("mobilenet", "resnet18", "effnet_b0")
MODEL_SPECS = {
    # name                 experts            fusion  temp  lambda_div
    "agef_sharp":        (EXPERT_NAMES,       "gate", 1.0,  0.05),
    "agef_soft":         (EXPERT_NAMES,       "gate", 2.0,  0.05),
    "agef_nodiv":        (EXPERT_NAMES,       "gate", 1.0,  0.0),
    "avg_joint":         (EXPERT_NAMES,       "avg",  1.0,  0.0),
    "single_mobilenet":  (("mobilenet",),     "avg",  1.0,  0.0),
    "single_resnet18":   (("resnet18",),      "avg",  1.0,  0.0),
    "single_effnet_b0":  (("effnet_b0",),     "avg",  1.0,  0.0),
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


# -----------------------------------------------------------------------------
# 1. Data
# -----------------------------------------------------------------------------
def make_transform(img):
    return transforms.Compose([
        transforms.Resize((img, img)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


class IndexedSubset(Dataset):
    """Subset of an ImageFolder with labels remapped to 0..K-1 (unknown -> -1)."""
    def __init__(self, base, indices, remap):
        self.base, self.indices, self.remap = base, np.asarray(indices), remap

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        path, t = self.base.samples[self.indices[i]]
        img = self.base.loader(path)
        if self.base.transform is not None:
            img = self.base.transform(img)
        return img, self.remap.get(t, -1)


def _strat_split(idx, y, test_size, seed):
    try:
        return train_test_split(idx, test_size=test_size, stratify=y, random_state=seed)
    except ValueError:  # a class too small to stratify
        return train_test_split(idx, test_size=test_size, random_state=seed)


def image_keys(path):
    """file MD5, decoded grey-level pixel MD5, and 256-bit dHash (17x16) as hex."""
    import hashlib
    from PIL import Image
    b = open(path, "rb").read()
    with Image.open(path) as im:
        g = im.convert("L")
        pmd5 = hashlib.md5(g.tobytes() + str(g.size).encode()).hexdigest()
        a = np.asarray(g.resize((17, 16), Image.BILINEAR), dtype=np.int16)
    return hashlib.md5(b).hexdigest(), pmd5, np.packbits((a[:, 1:] > a[:, :-1]).flatten()).tobytes().hex()


def compute_groups(base, cache_path=None, key_types=(0, 1), unsplittable_share=0.9, min_class_images=10,
                   near_bits=None, near_cap=50):
    """Union of exact-key equalities (+ optional capped within-family near-duplicate merging) -> duplicate groups.
    Returns dict(groups, drop, exclude_classes, stats, protocol)."""
    keys = None
    if cache_path and os.path.exists(cache_path):
        c = json.load(open(cache_path))
        if c.get("n") == len(base.samples): keys = c["keys"]
    if keys is None:
        keys = []
        for i, (p, _) in enumerate(base.samples):
            keys.append(image_keys(p))
            if (i + 1) % 5000 == 0: print(f"    hashed {i + 1}/{len(base.samples)}", flush=True)
        if cache_path:
            os.makedirs(os.path.dirname(cache_path), exist_ok=True)
            json.dump(dict(n=len(base.samples), keys=keys), open(cache_path, "w"))
    n = len(keys); parent = np.arange(n)
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for t in key_types:
        first = {}
        for i, k in enumerate(keys):
            j = first.setdefault(k[t], i)
            if j != i:
                ri, rj = find(i), find(j)
                if ri != rj: parent[ri] = rj
    targets = np.array([t for _, t in base.samples])
    n_near_merges = 0
    if near_bits is not None:
        POP = np.array([bin(i).count("1") for i in range(256)], np.uint8)
        hb = np.stack([np.frombuffer(bytes.fromhex(k[2]), dtype=np.uint8) for k in keys])
        size = np.zeros(n, int)
        for i in range(n): size[find(i)] += 1
        edges = []
        for c in np.unique(targets):
            idx = np.where(targets == c)[0]
            for a0 in range(0, len(idx), 256):
                q = idx[a0:a0 + 256]
                d = POP[hb[q][:, None, :] ^ hb[idx][None, :, :]].sum(-1)
                ii, jj = np.where(d <= near_bits)
                for u, v in zip(ii, jj):
                    if q[u] < idx[v]: edges.append((int(d[u, v]), int(q[u]), int(idx[v])))
        edges.sort()
        for _, i, j in edges:
            ri, rj = find(i), find(j)
            if ri != rj and size[ri] + size[rj] <= near_cap:
                parent[ri] = rj; size[rj] += size[ri]; n_near_merges += 1
    roots = np.array([find(i) for i in range(n)])
    _, groups = np.unique(roots, return_inverse=True)
    fam_per_group = pd.Series(targets).groupby(groups).nunique()
    conflict = fam_per_group[fam_per_group > 1].index.values
    drop = np.isin(groups, conflict)
    sizes = np.bincount(groups)
    exclude = []
    for c in range(len(base.classes)):
        m = (targets == c) & ~drop
        if m.sum() < min_class_images or (m.any() and np.bincount(groups[m]).max() > unsplittable_share * m.sum()):
            exclude.append(c)
    stats = dict(n_images=n, n_groups=int(len(sizes)), n_images_in_multi_groups=int((sizes[groups] > 1).sum()),
                 largest_group=int(sizes.max()), n_conflict_groups=int(len(conflict)), n_dropped=int(drop.sum()),
                 dropped_by_class={base.classes[c]: int(k) for c, k in zip(*np.unique(targets[drop], return_counts=True))},
                 excluded_classes=[base.classes[c] for c in exclude],
                 largest_group_share_by_class={base.classes[c]: round(float(np.bincount(groups[(targets == c) & ~drop]).max()
                                               / max(((targets == c) & ~drop).sum(), 1)), 3) if ((targets == c) & ~drop).any() else None
                                               for c in range(len(base.classes))},
                 near_dup=None if near_bits is None else dict(bits=near_bits, cap=near_cap, merges=n_near_merges))
    prot = PROTOCOL if near_bits is None else f"{PROTOCOL}+near-L3-{near_bits}bit-cap{near_cap}"
    return dict(groups=groups, drop=drop, exclude_classes=exclude, stats=stats, protocol=prot)


def make_split(base, seed, cfg, G=None):
    """Class-disjoint known/unknown split + 4-way split of known data (70/10/10/10).
    With G (from compute_groups): duplicate groups are kept within one part and label-conflict groups are removed."""
    classes = base.classes
    targets = np.array([t for _, t in base.samples])
    never = {n.lower() for n in cfg["NEVER_UNKNOWN"]}
    eligible = [c for c in range(len(classes)) if classes[c].lower() not in never]
    protected = [classes[c] for c in range(len(classes)) if classes[c].lower() in never]
    if protected:
        print(f"  classes always kept known (benign/other): {protected}")
    n_unk = max(1, int(round(len(classes) * cfg["UNKNOWN_RATIO"])))
    rng = np.random.RandomState(seed)
    unk = sorted(rng.choice(eligible, n_unk, replace=False).tolist())
    known = [c for c in range(len(classes)) if c not in unk]
    excluded = [] if G is None else list(G.get("exclude_classes", []))
    if excluded:
        unk = [c for c in unk if c not in excluded]; known = [c for c in known if c not in excluded]
    keep = np.ones(len(targets), bool) if G is None else ~G["drop"]
    known_idx = np.where(np.isin(targets, known) & keep)[0]
    unk_idx = np.where(np.isin(targets, unk) & keep)[0]
    if G is None:
        f_tr, f_va, f_ca, f_te = cfg["FRACS"]
        tr, rest = _strat_split(known_idx, targets[known_idx], 1 - f_tr, seed)
        va, rest2 = _strat_split(rest, targets[rest], (f_ca + f_te) / (f_va + f_ca + f_te), seed)
        ca, te = _strat_split(rest2, targets[rest2], f_te / (f_ca + f_te), seed)
    else:
        from sklearn.model_selection import StratifiedGroupKFold
        fold = np.full(len(known_idx), -1)
        sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=seed)
        for f, (_, te_) in enumerate(sgkf.split(known_idx, targets[known_idx], G["groups"][known_idx])):
            fold[te_] = f
        tr, va, ca, te = (known_idx[fold <= 6], known_idx[fold == 7], known_idx[fold == 8], known_idx[fold == 9])
        missing = [classes[c] for c in known if not np.isin(c, targets[tr])]
        assert not missing, f"known classes without training images after grouping: {missing}"
        for nm, part in [("val", va), ("calib", ca), ("test", te)]:
            lack = [classes[c] for c in known if not np.isin(c, targets[part])]
            if lack: print(f"  warning: seed {seed}: no {nm} images for {lack} (group sizes)")
    remap = {c: i for i, c in enumerate(known)}
    out = dict(known=known, unknown=unk, remap=remap,
               known_names=[classes[c] for c in known], unknown_names=[classes[c] for c in unk],
               train=tr, val=va, calib=ca, test=te, unk=unk_idx,
               protocol=G.get("protocol", PROTOCOL) if G is not None else "random-v2")
    if G is not None:
        out["n_dropped_conflict"] = int(G["drop"].sum())
        out["excluded_classes"] = [classes[c] for c in excluded]
    return out


def loader(base, idx, remap, cfg, shuffle):
    return DataLoader(IndexedSubset(base, idx, remap), batch_size=cfg["BATCH"], shuffle=shuffle,
                      num_workers=cfg["WORKERS"], pin_memory=torch.cuda.is_available(),
                      persistent_workers=False)


# -----------------------------------------------------------------------------
# 2. Models
# -----------------------------------------------------------------------------
class Expert(nn.Module):
    """Backbone returning penultimate features + its own linear head."""
    def __init__(self, name, num_classes, pretrained):
        super().__init__()
        w = "IMAGENET1K_V1" if pretrained else None
        if name == "mobilenet":
            m = models.mobilenet_v3_small(weights=w); d = m.classifier[3].in_features; m.classifier[3] = nn.Identity()
        elif name == "resnet18":
            m = models.resnet18(weights=w); d = m.fc.in_features; m.fc = nn.Identity()
        elif name == "effnet_b0":
            m = models.efficientnet_b0(weights=w); d = m.classifier[1].in_features; m.classifier[1] = nn.Identity()
        else:
            raise ValueError(name)
        self.name, self.backbone, self.head, self.feat_dim = name, m, nn.Linear(d, num_classes), d

    def forward(self, x):
        f = self.backbone(x)
        return f, self.head(f)


class MultiExpert(nn.Module):
    """AGEF-Net (fusion='gate') or uniform average (fusion='avg'). One expert = single CNN."""
    def __init__(self, expert_names, num_classes, fusion="gate", temp=1.0, pretrained=True):
        super().__init__()
        self.experts = nn.ModuleList([Expert(n, num_classes, pretrained) for n in expert_names])
        self.E, self.fusion, self.temp = len(expert_names), fusion, temp
        if fusion == "gate":
            self.gate = nn.Sequential(nn.Linear(num_classes * self.E, 128), nn.ReLU(), nn.Linear(128, self.E))

    def forward(self, x):
        feats, logits = zip(*[e(x) for e in self.experts])
        stacked = torch.stack(logits, 1)                       # B,E,C
        if self.fusion == "gate":
            w = F.softmax(self.gate(stacked.flatten(1)) / self.temp, dim=1)
        else:
            w = torch.full((x.size(0), self.E), 1.0 / self.E, device=x.device)
        fused = (w.unsqueeze(-1) * stacked).sum(1)
        # per-expert L2 normalisation so no single backbone dominates KNN distances
        return dict(logits=fused, expert_logits=stacked, weights=w,
                    feats=torch.cat([F.normalize(f.float(), dim=1) for f in feats], 1))


# -----------------------------------------------------------------------------
# 3. Training (model selection on known-class val accuracy only)
# -----------------------------------------------------------------------------
def train(model, tr_ld, va_ld, cfg, lambda_div, log_prefix=""):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["LR"], weight_decay=cfg["WD"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["EPOCHS"])
    use_amp = cfg["AMP"] and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    best, best_state, hist = -1.0, None, []
    for ep in range(cfg["EPOCHS"]):
        model.train(); t0 = time.time(); tl, n = 0.0, 0
        for xb, yb in tr_ld:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                out = model(xb)
                loss = F.cross_entropy(out["logits"], yb)
                if lambda_div > 0 and model.E > 1 and model.fusion == "gate":
                    wbar = out["weights"].float().mean(0)
                    loss = loss + lambda_div * (1.0 - (-(wbar * torch.log(wbar + 1e-8)).sum()))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item() * xb.size(0); n += xb.size(0)
        sched.step()
        acc = evaluate_acc(model, va_ld, use_amp)
        hist.append(dict(epoch=ep, train_loss=tl / n, val_acc=acc, sec=time.time() - t0))
        print(f"{log_prefix} ep {ep:2d} | loss {tl / n:.4f} | val acc {acc:.2f}% | {time.time() - t0:.0f}s", flush=True)
        if acc > best:
            best, best_state = acc, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, best, hist


@torch.no_grad()
def evaluate_acc(model, ld, use_amp=False):
    model.eval(); c = t = 0
    for xb, yb in ld:
        with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
            p = model(xb.to(DEVICE))["logits"].argmax(1).cpu()
        c += (p == yb).sum().item(); t += yb.numel()
    return 100.0 * c / max(t, 1)


@torch.no_grad()
def collect(model, ld):
    """Return float64 numpy outputs for every sample in the loader (fp32 forward)."""
    model.eval(); acc = {k: [] for k in ("logits", "expert_logits", "weights", "feats", "y")}
    for xb, yb in ld:
        out = model(xb.to(DEVICE))
        for k in ("logits", "expert_logits", "weights", "feats"):
            acc[k].append(out[k].float().cpu().numpy().astype(np.float64))
        acc["y"].append(yb.numpy())
    return {k: np.concatenate(v) for k, v in acc.items()}


# -----------------------------------------------------------------------------
# 4. Scores (higher = more likely unknown)
# -----------------------------------------------------------------------------
def entropy(p):
    return -(p * np.log(np.clip(p, 1e-300, 1.0))).sum(-1)


def base_scores(o, is_deep_ensemble=False):
    """o: dict with logits [N,C], expert_logits [N,E,C], weights [N,E]."""
    if is_deep_ensemble:
        pe = softmax(o["expert_logits"], axis=-1)              # N,E,C
        p = pe.mean(1)
        logits = o["expert_logits"].mean(1)
    else:
        logits = o["logits"]; p = softmax(logits, axis=-1)
        pe = softmax(o["expert_logits"], axis=-1)
    s = OrderedDict()
    s["msp"] = -p.max(1)
    s["entropy"] = entropy(p)
    s["maxlogit"] = -logits.max(1)
    s["energy"] = -logsumexp(logits, axis=1)
    if pe.shape[1] > 1:
        s["mutual_info"] = entropy(pe.mean(1)) - entropy(pe).mean(1)
    if (not is_deep_ensemble) and o["weights"].shape[1] > 1 and np.ptp(o["weights"]) > 1e-9:
        s["gate_entropy"] = entropy(o["weights"])
    return s


class KNNScorer:
    """Sun et al. (2022): negative cosine similarity to k-th nearest training feature."""
    def __init__(self, train_feats, k, max_train, seed):
        rng = np.random.RandomState(seed)
        if len(train_feats) > max_train:
            train_feats = train_feats[rng.choice(len(train_feats), max_train, replace=False)]
        self.bank = torch.tensor(train_feats / np.linalg.norm(train_feats, axis=1, keepdims=True), dtype=torch.float32)
        self.k = min(k, len(self.bank))

    def __call__(self, feats):
        q = torch.tensor(feats / np.linalg.norm(feats, axis=1, keepdims=True), dtype=torch.float32)
        out = []
        for i in range(0, len(q), 1024):
            sims = q[i:i + 1024] @ self.bank.T
            out.append(-sims.topk(self.k, dim=1).values[:, -1])
        return torch.cat(out).numpy().astype(np.float64)


class OpenMax:
    """Bendale & Boult (2016) on fused logits: per-class MAV, Weibull fit on the
    largest distances of correctly classified training samples, top-alpha revision."""
    def __init__(self, train_logits, train_y, tail, alpha):
        from scipy.stats import weibull_min
        self.C = train_logits.shape[1]; self.alpha = min(alpha, self.C)
        self.mav = np.zeros((self.C, self.C)); self.params = []
        pred = train_logits.argmax(1)
        for c in range(self.C):
            av = train_logits[(train_y == c) & (pred == c)]
            if len(av) == 0:
                av = train_logits[train_y == c]
            self.mav[c] = av.mean(0)
            d = np.linalg.norm(av - self.mav[c], axis=1)
            t = np.sort(d)[-min(tail, len(d)):]
            if len(t) < 3 or t.std() < 1e-9:
                self.params.append(None)
            else:
                self.params.append(weibull_min.fit(t, floc=0))
        self._cdf = weibull_min.cdf

    def __call__(self, logits):
        N = len(logits); out = np.zeros(N)
        ranks = np.argsort(-logits, axis=1)[:, :self.alpha]
        wts = np.array([(self.alpha - i) / self.alpha for i in range(self.alpha)])
        for n in range(N):
            v = logits[n].copy(); unk = 0.0
            for r, c in enumerate(ranks[n]):
                prm = self.params[c]
                cdf = 0.0 if prm is None else self._cdf(np.linalg.norm(logits[n] - self.mav[c]), *prm)
                om = wts[r] * cdf
                unk += v[c] * om
                v[c] = v[c] * (1 - om)
            z = np.concatenate([[unk], v])
            out[n] = softmax(z)[0]                               # P(unknown)
        return out


# -----------------------------------------------------------------------------
# 5. Metrics
# -----------------------------------------------------------------------------
def fpr_at_95tpr(s_known, s_unk):
    """Known = positive (accepted if score <= thr). FPR = unknowns accepted."""
    thr = np.quantile(s_known, 0.95)
    return float((s_unk <= thr).mean())


def operating_point(s_calib, s_test, s_unk, target_far):
    thr = np.quantile(s_calib, 1 - target_far)
    return dict(
        thr=float(thr),
        zdr=float((s_unk > thr).mean()),
        far_test=float((s_test > thr).mean()),
        far_calib=float((s_calib > thr).mean()),
        tie_frac_calib=float(np.isclose(s_calib, thr, rtol=0, atol=1e-12).mean()),
    )


def threshold_free(s_known_test, s_unk):
    y = np.r_[np.zeros(len(s_known_test)), np.ones(len(s_unk))]
    s = np.r_[s_known_test, s_unk]
    return dict(auroc=float(roc_auc_score(y, s)), aupr_out=float(average_precision_score(y, s)),
                fpr95=fpr_at_95tpr(s_known_test, s_unk))


def four_tag(conf_score, unc_score, tau_c, tau_u):
    """conf_score / unc_score are 'higher = more unknown'. Paper's Eq. 7 mapping:
    0 Auto-block (confident & certain), 1 Quarantine (unconfident & uncertain),
    2 Review (confident & uncertain), 3 Log (unconfident & certain)."""
    lowconf, highunc = conf_score > tau_c, unc_score > tau_u
    return np.where(~lowconf & ~highunc, 0, np.where(lowconf & highunc, 1, np.where(~lowconf & highunc, 2, 3)))


def triage_report(sc_calib, sc_test, sc_unk, axes, target_far):
    a, b = axes
    tau_a = np.quantile(sc_calib[a], 1 - target_far)
    tau_b = np.quantile(sc_calib[b], 1 - target_far)
    tk = four_tag(sc_test[a], sc_test[b], tau_a, tau_b)
    tu = four_tag(sc_unk[a], sc_unk[b], tau_a, tau_b)
    r = dict(tau_a=float(tau_a), tau_b=float(tau_b),
             zdr=float(np.isin(tu, [1, 2]).mean()), far_test=float(np.isin(tk, [1, 2]).mean()),
             wr_test=float(np.isin(tk, [0, 3]).mean()))
    for t, nm in enumerate(["autoblock", "quarantine", "review", "log"]):
        r[f"known_{nm}"] = float((tk == t).mean()); r[f"unk_{nm}"] = float((tu == t).mean())
    return r


# -----------------------------------------------------------------------------
# 6. Dynamic early exit (honest version)
# -----------------------------------------------------------------------------
def measure_expert_costs(model, cfg):
    """FLOPs per image (torch flop counter) and mean latency per 32-image batch, per expert + gate."""
    from torch.utils.flop_counter import FlopCounterMode
    model.eval(); x1 = torch.randn(1, 3, cfg["IMG"], cfg["IMG"], device=DEVICE)
    xb = torch.randn(cfg["BATCH"], 3, cfg["IMG"], cfg["IMG"], device=DEVICE)
    costs = {}
    with torch.no_grad():
        for e in model.experts:
            fc = FlopCounterMode(display=False)
            with fc:
                e(x1)
            for _ in range(3):
                e(xb)
            if DEVICE.type == "cuda": torch.cuda.synchronize()
            t0 = time.time()
            for _ in range(10):
                e(xb)
            if DEVICE.type == "cuda": torch.cuda.synchronize()
            costs[e.name] = dict(gflops=fc.get_total_flops() / 1e9, ms_per_batch=(time.time() - t0) / 10 * 1e3)
    return costs


def dynamic_eval(o_cal, o_test, o_unk, costs, cfg):
    """Exit after MobileNet (expert 0) when its own MSP >= theta; theta chosen as a
    quantile of calib MobileNet-MSP so that a target fraction of known traffic exits.
    The detection score is entropy of whichever output is used; its threshold is
    re-calibrated on calib under the same policy."""
    rows = []
    f_m, f_rest = costs["mobilenet"]["gflops"], sum(v["gflops"] for k, v in costs.items() if k != "mobilenet")
    t_m, t_rest = costs["mobilenet"]["ms_per_batch"], sum(v["ms_per_batch"] for k, v in costs.items() if k != "mobilenet")

    def policy(o, theta):
        pm = softmax(o["expert_logits"][:, 0], axis=-1)
        exit_ = pm.max(1) >= theta
        p = np.where(exit_[:, None], pm, softmax(o["logits"], axis=-1))
        return entropy(p), exit_

    mcal = softmax(o_cal["expert_logits"][:, 0], axis=-1).max(1)
    for q in cfg["DYNAMIC_EXIT_QUANTILES"]:
        theta = np.inf if q == 0 else np.quantile(mcal, 1 - q)
        s_cal, _ = policy(o_cal, theta)
        s_te, ex_te = policy(o_test, theta)
        s_un, ex_un = policy(o_unk, theta)
        op = operating_point(s_cal, s_te, s_un, cfg["TARGET_FAR"])
        exit_rate = np.r_[ex_te, ex_un].mean()
        rows.append(dict(target_exit=q, theta=float(theta), exit_rate_known=float(ex_te.mean()),
                         exit_rate_unknown=float(ex_un.mean()),
                         gflops_per_img=f_m + (1 - exit_rate) * f_rest,
                         gflops_static=f_m + f_rest,
                         est_ms_per_batch=t_m + (1 - exit_rate) * t_rest,
                         ms_static=t_m + t_rest, **op))
    return rows


# -----------------------------------------------------------------------------
# 7. One (dataset, seed) run
# -----------------------------------------------------------------------------
def run_split(ds_name, base, seed, cfg):
    out_dir = os.path.join(cfg["OUT_DIR"], ds_name, f"seed{seed}")
    os.makedirs(out_dir, exist_ok=True)
    sp = make_split(base, seed, cfg, cfg.get("_GROUPS", {}).get(ds_name))
    json.dump({k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in sp.items() if k != "remap"},
              open(os.path.join(out_dir, "split.json"), "w"), indent=1)
    print(f"\n[{ds_name} seed {seed}] known={len(sp['known'])} unknown={sp['unknown_names']} "
          f"| train {len(sp['train'])} val {len(sp['val'])} calib {len(sp['calib'])} test {len(sp['test'])} unk {len(sp['unk'])}")

    lds = {k: loader(base, sp[k], sp["remap"], cfg, shuffle=(k == "train")) for k in ("train", "val")}
    eval_lds = {k: loader(base, sp[k], sp["remap"], cfg, shuffle=False) for k in ("train", "calib", "test", "unk")}
    K = len(sp["known"])
    outputs = {}

    for mname in cfg["MODELS"]:
        npz = os.path.join(out_dir, f"{mname}.npz")
        if os.path.exists(npz):
            d = np.load(npz, allow_pickle=True)
            outputs[mname] = {part: {k: d[f"{part}__{k}"] for k in ("logits", "expert_logits", "weights", "feats", "y")}
                              for part in ("train", "calib", "test", "unk")}
            outputs[mname]["_meta"] = json.loads(str(d["meta"]))
            print(f"  [{mname}] loaded cached outputs"); continue
        experts, fusion, temp, lam = MODEL_SPECS[mname]
        seed_everything(seed * 1000 + zlib.crc32(mname.encode()) % 997)
        model = MultiExpert(experts, K, fusion, temp, cfg["PRETRAINED"]).to(DEVICE)
        model, best_val, hist = train(model, lds["train"], lds["val"], cfg, lam, log_prefix=f"  [{mname}]")
        o = {part: collect(model, ld) for part, ld in eval_lds.items()}
        meta = dict(best_val_acc=best_val, history=hist,
                    test_acc=float(100 * (o["test"]["logits"].argmax(1) == o["test"]["y"]).mean()))
        if cfg["RUN_DYNAMIC"] and fusion == "gate":
            meta["costs"] = measure_expert_costs(model, cfg)
        np.savez_compressed(npz, meta=json.dumps(meta),
                            **{f"{p}__{k}": v for p, d in o.items() for k, v in d.items()})
        o["_meta"] = meta; outputs[mname] = o
        del model
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

    singles = [f"single_{n}" for n in EXPERT_NAMES]
    if cfg["DEEP_ENSEMBLE"] and all(s in outputs for s in singles):
        de = {}
        for part in ("train", "calib", "test", "unk"):
            de[part] = dict(
                expert_logits=np.concatenate([outputs[s][part]["expert_logits"] for s in singles], 1),
                feats=np.concatenate([outputs[s][part]["feats"] for s in singles], 1),
                y=outputs[singles[0]][part]["y"])
            de[part]["logits"] = de[part]["expert_logits"].mean(1)
            de[part]["weights"] = np.full((len(de[part]["y"]), 3), 1 / 3)
        p = softmax(de["test"]["expert_logits"], -1).mean(1)
        de["_meta"] = dict(test_acc=float(100 * (p.argmax(1) == de["test"]["y"]).mean()), deep_ensemble=True)
        outputs["deep_ensemble"] = de

    # ---- score + metric every model -------------------------------------------------
    rows, tri_rows, dyn_rows, cls_rows = [], [], [], []
    targets = np.array([t for _, t in base.samples])
    unk_family = np.array([base.classes[t] for t in targets[sp["unk"]]])       # loader order == sp["unk"] order
    for mname, o in outputs.items():
        is_de = mname == "deep_ensemble"
        sc = {part: base_scores(o[part], is_de) for part in ("calib", "test", "unk")}
        knn = KNNScorer(o["train"]["feats"], cfg["KNN_K"], cfg["KNN_MAX_TRAIN"], seed)
        for part in ("calib", "test", "unk"):
            sc[part]["knn"] = knn(o[part]["feats"])
        om = OpenMax(o["train"]["logits"], o["train"]["y"], cfg["OPENMAX_TAIL"], cfg["OPENMAX_ALPHA"])
        for part in ("calib", "test", "unk"):
            sc[part]["openmax"] = om(o[part]["logits"])
        for sname in sc["test"]:
            r = dict(dataset=ds_name, seed=seed, model=mname, score=sname,
                     test_acc=o["_meta"]["test_acc"], n_unknown_classes=len(sp["unknown"]))
            r.update(threshold_free(sc["test"][sname], sc["unk"][sname]))
            r.update(operating_point(sc["calib"][sname], sc["test"][sname], sc["unk"][sname], cfg["TARGET_FAR"]))
            rows.append(r)
        # per-class alarm rate: recall for each held-out family, false-alarm rate for each known class
        test_cls = np.array([sp["known_names"][y] for y in o["test"]["y"]])
        for sname in ("msp", "entropy", "energy", "knn"):
            if sname not in sc["test"]:
                continue
            thr = np.quantile(sc["calib"][sname], 1 - cfg["TARGET_FAR"])
            for fam in np.unique(unk_family):
                m_ = unk_family == fam
                cls_rows.append(dict(dataset=ds_name, seed=seed, model=mname, score=sname, cls=fam, role="unknown",
                                     n=int(m_.sum()), alarm_rate=float((sc["unk"][sname][m_] > thr).mean())))
            for kc in np.unique(test_cls):
                m_ = test_cls == kc
                cls_rows.append(dict(dataset=ds_name, seed=seed, model=mname, score=sname, cls=kc, role="known",
                                     n=int(m_.sum()), alarm_rate=float((sc["test"][sname][m_] > thr).mean())))
        tri = triage_report(sc["calib"], sc["test"], sc["unk"], cfg["TRIAGE_AXES"], cfg["TARGET_FAR"])
        tri_rows.append(dict(dataset=ds_name, seed=seed, model=mname, axes="/".join(cfg["TRIAGE_AXES"]), **tri))
        if cfg["RUN_DYNAMIC"] and "costs" in o["_meta"]:
            for d in dynamic_eval(o["calib"], o["test"], o["unk"], o["_meta"]["costs"], cfg):
                dyn_rows.append(dict(dataset=ds_name, seed=seed, model=mname, **d))
    return rows, tri_rows, dyn_rows, cls_rows


# -----------------------------------------------------------------------------
# 8. Aggregation across splits
# -----------------------------------------------------------------------------
def boot_ci(x, n_boot, seed=0):
    x = np.asarray(x, float)
    if len(x) < 2:
        return (np.nan, np.nan)
    rng = np.random.RandomState(seed)
    m = rng.choice(x, (n_boot, len(x)), replace=True).mean(1)
    return tuple(np.percentile(m, [2.5, 97.5]))


def summarise(raw, cfg):
    metrics = ["auroc", "aupr_out", "fpr95", "zdr", "far_test", "tie_frac_calib", "test_acc"]
    out = []
    for (d, m, s), g in raw.groupby(["dataset", "model", "score"]):
        r = dict(dataset=d, model=m, score=s, n_splits=len(g))
        for k in metrics:
            r[f"{k}_mean"], r[f"{k}_std"] = g[k].mean(), g[k].std(ddof=1) if len(g) > 1 else np.nan
        r["auroc_ci_lo"], r["auroc_ci_hi"] = boot_ci(g["auroc"], cfg["N_BOOT"])
        r["zdr_ci_lo"], r["zdr_ci_hi"] = boot_ci(g["zdr"], cfg["N_BOOT"])
        out.append(r)
    return pd.DataFrame(out)


def paired_tests(raw, cfg, ref=("agef_sharp", "entropy")):
    """Reference (model, score) vs every other (model, score), paired over split seeds."""
    out = []
    for d, gd in raw.groupby("dataset"):
        a = gd[(gd.model == ref[0]) & (gd.score == ref[1])].set_index("seed")
        if a.empty:
            continue
        for (m, s), gb in gd.groupby(["model", "score"]):
            if (m, s) == ref:
                continue
            b = gb.set_index("seed")
            common = a.index.intersection(b.index)
            if len(common) < 2:
                continue
            for k in ("auroc", "zdr"):
                diff = a.loc[common, k].values - b.loc[common, k].values
                t = stats.ttest_rel(a.loc[common, k], b.loc[common, k])
                try:
                    w = stats.wilcoxon(diff).pvalue if np.any(diff != 0) else 1.0
                except ValueError:
                    w = np.nan
                lo, hi = boot_ci(diff, cfg["N_BOOT"])
                out.append(dict(dataset=d, reference=f"{ref[0]}:{ref[1]}", other=f"{m}:{s}", metric=k,
                                n=len(common), mean_diff=diff.mean(), diff_ci_lo=lo, diff_ci_hi=hi,
                                t_p=t.pvalue, wilcoxon_p=w))
    return pd.DataFrame(out)


# -----------------------------------------------------------------------------
# 9. Smoke test data (synthetic, CPU) — verifies the pipeline end to end
# -----------------------------------------------------------------------------
def build_synthetic(root, n_classes=6, n_per=40, img=32, seed=0):
    from PIL import Image
    rng = np.random.RandomState(seed)
    for c in range(n_classes):
        os.makedirs(os.path.join(root, f"fam{c}"), exist_ok=True)
        base = rng.rand(img, img)
        for i in range(n_per):
            a = np.clip(base + 0.25 * rng.randn(img, img), 0, 1)
            Image.fromarray((a * 255).astype(np.uint8), "L").save(os.path.join(root, f"fam{c}", f"{i}.png"))
    os.makedirs(os.path.join(root, "Other"), exist_ok=True)
    for i in range(n_per):
        Image.fromarray((rng.rand(img, img) * 255).astype(np.uint8), "L").save(os.path.join(root, "Other", f"{i}.png"))


# -----------------------------------------------------------------------------
# 10. Main
# -----------------------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--datasets", nargs="*", help="subset of CFG['DATASETS'] keys")
    ap.add_argument("--path", nargs="*", help="override paths as Name=/path")
    ap.add_argument("--seeds", nargs="*", type=int)
    ap.add_argument("--models", nargs="*")
    ap.add_argument("--epochs", type=int)
    ap.add_argument("--img", type=int)
    ap.add_argument("--batch", type=int)
    ap.add_argument("--workers", type=int)
    ap.add_argument("--out")
    ap.add_argument("--no-pretrained", action="store_true")
    ap.add_argument("--no-dynamic", action="store_true")
    ap.add_argument("--no-grouping", action="store_true", help="old random splits (protocol v2)")
    ap.add_argument("--smoke", action="store_true", help="synthetic CPU run to verify the pipeline")
    ap.add_argument("--resume-from", help="folder with a previous OUT_DIR (e.g. an attached Kaggle output); cached runs are copied and skipped")
    ap.add_argument("--max-hours", type=float, default=None, help="do not start a new split after this many hours")
    args, _ = ap.parse_known_args()

    cfg = copy.deepcopy(CFG)
    if args.smoke:
        root = "/tmp/auditx_smoke_data"
        if not os.path.exists(root):
            build_synthetic(root)
        cfg.update(DATASETS=OrderedDict(Synthetic=root), NEVER_UNKNOWN=["other"], SPLIT_SEEDS=[0, 1], EPOCHS=2, IMG=64, BATCH=16,
                   WORKERS=0, PRETRAINED=False, OUT_DIR="/tmp/auditx_smoke_out", N_BOOT=500,
                   KNN_K=5, OPENMAX_TAIL=5, UNKNOWN_RATIO=0.34)
    if args.path:
        for kv in args.path:
            k, v = kv.split("=", 1); cfg["DATASETS"][k] = v
    if args.datasets:
        cfg["DATASETS"] = OrderedDict((k, cfg["DATASETS"][k]) for k in args.datasets)
    if args.seeds: cfg["SPLIT_SEEDS"] = args.seeds
    if args.models: cfg["MODELS"] = args.models
    if args.epochs: cfg["EPOCHS"] = args.epochs
    if args.img: cfg["IMG"] = args.img
    if args.batch: cfg["BATCH"] = args.batch
    if args.workers is not None: cfg["WORKERS"] = args.workers
    if args.out: cfg["OUT_DIR"] = args.out
    if args.no_pretrained: cfg["PRETRAINED"] = False
    if args.no_dynamic: cfg["RUN_DYNAMIC"] = False
    if args.no_grouping: cfg["GROUPING"] = False

    os.makedirs(cfg["OUT_DIR"], exist_ok=True)
    if args.resume_from and os.path.isdir(args.resume_from):
        import shutil
        n_copied = 0
        for dp, _, fns in os.walk(args.resume_from):
            for fn in fns:
                if fn.endswith(".npz") or fn == "split.json":
                    rel = os.path.relpath(os.path.join(dp, fn), args.resume_from)
                    dst = os.path.join(cfg["OUT_DIR"], rel)
                    if not os.path.exists(dst):
                        os.makedirs(os.path.dirname(dst), exist_ok=True)
                        shutil.copy2(os.path.join(dp, fn), dst); n_copied += 1
        print(f"Resumed {n_copied} cached files from {args.resume_from}")
    t_start = time.time()
    json.dump({k: (list(v.items()) if isinstance(v, OrderedDict) else v) for k, v in cfg.items()},
              open(os.path.join(cfg["OUT_DIR"], "config.json"), "w"), indent=1)
    print(f"Device: {DEVICE} | torch {torch.__version__} | out: {cfg['OUT_DIR']}")

    all_rows, all_tri, all_dyn, all_cls = [], [], [], []
    for ds_name, path in cfg["DATASETS"].items():
        if not path or not os.path.isdir(path):
            print(f"!! skipping {ds_name}: path not found ({path})"); continue
        base = datasets.ImageFolder(path, transform=make_transform(cfg["IMG"]))
        print(f"\n===== {ds_name}: {len(base)} images, {len(base.classes)} classes | protocol {PROTOCOL if cfg['GROUPING'] else 'random-v2'} =====")
        if cfg["GROUPING"]:
            near = ds_name in cfg.get("NEAR_DUP_DATASETS", ())
            G = compute_groups(base, os.path.join(cfg["OUT_DIR"], ds_name, "image_keys.json"), cfg["GROUP_KEYS"],
                               cfg["UNSPLITTABLE_SHARE"], cfg["MIN_CLASS_IMAGES"],
                               cfg["NEAR_DUP_BITS"] if near else None, cfg["NEAR_DUP_CAP"])
            print("  split protocol:", G["protocol"])
            cfg.setdefault("_GROUPS", {})[ds_name] = G
            json.dump(G["stats"], open(os.path.join(cfg["OUT_DIR"], ds_name, "groups_stats.json"), "w"), indent=1)
            print("  duplicate groups:", json.dumps({k: v for k, v in G["stats"].items() if k not in ("dropped_by_class", "largest_group_share_by_class")}),
                  "| dropped (label conflicts):", G["stats"]["dropped_by_class"], "| excluded classes:", G["stats"]["excluded_classes"])
        for seed in cfg["SPLIT_SEEDS"]:
            if args.max_hours and (time.time() - t_start) / 3600 > args.max_hours:
                print(f"!! time budget of {args.max_hours} h reached; not starting seed {seed}. "
                      f"Re-run with --resume-from to continue."); break
            r, t, d, c = run_split(ds_name, base, seed, cfg)
            all_cls += c
            pd.DataFrame(all_cls).to_csv(os.path.join(cfg["OUT_DIR"], "per_class_raw.csv"), index=False)
            print(f"  elapsed {(time.time() - t_start) / 3600:.2f} h", flush=True)
            all_rows += r; all_tri += t; all_dyn += d
            # write after every split so a timed-out session still leaves results
            pd.DataFrame(all_rows).to_csv(os.path.join(cfg["OUT_DIR"], "results_raw.csv"), index=False)
            pd.DataFrame(all_tri).to_csv(os.path.join(cfg["OUT_DIR"], "triage_raw.csv"), index=False)
            if all_dyn:
                pd.DataFrame(all_dyn).to_csv(os.path.join(cfg["OUT_DIR"], "dynamic_raw.csv"), index=False)

    if not all_rows:
        print("No results."); return
    raw = pd.DataFrame(all_rows)
    summ = summarise(raw, cfg); summ.to_csv(os.path.join(cfg["OUT_DIR"], "summary.csv"), index=False)
    paired_tests(raw, cfg).to_csv(os.path.join(cfg["OUT_DIR"], "paired_tests.csv"), index=False)
    tri = pd.DataFrame(all_tri)
    tri.drop(columns=["axes", "seed"]).groupby(["dataset", "model"]).agg(["mean", "std"]).to_csv(os.path.join(cfg["OUT_DIR"], "triage_summary.csv"))
    if all_dyn:
        dyn = pd.DataFrame(all_dyn)
        dyn.groupby(["dataset", "model", "target_exit"]).mean(numeric_only=True).to_csv(
            os.path.join(cfg["OUT_DIR"], "dynamic_summary.csv"))

    pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
    show = summ[summ.score.isin(["msp", "entropy", "energy", "knn", "openmax"])][
        ["dataset", "model", "score", "n_splits", "auroc_mean", "auroc_std", "fpr95_mean",
         "zdr_mean", "zdr_std", "far_test_mean", "tie_frac_calib_mean", "test_acc_mean"]]
    print("\n", show.round(4).to_string(index=False))
    print(f"\nAll outputs in {cfg['OUT_DIR']}")


if __name__ == "__main__":
    main()

In [ ]:
# Cell 2 - locate the cached runs and verify sample->class mapping
import os, sys, glob, json, re, warnings
import numpy as np, pandas as pd
from scipy import stats
from scipy.special import softmax
sys.path.insert(0, "/kaggle/working")
import importlib, auditx_eval_v2 as ax; importlib.reload(ax)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 300)

OUT = "/kaggle/working/auditx_v2_novelty"; os.makedirs(OUT, exist_ok=True)
DATASETS = ["Malimg", "MaleVis", "Dumpware10"]
REQUIRED_PROTOCOL = "grouped-L1-v3+near-L3-8bit-cap250"     # duplicate-aware splits; outputs of the old random splits are rejected
MODELS = ["agef_sharp", "avg_joint", "single_mobilenet", "single_resnet18", "single_effnet_b0"]
SINGLES = ["single_mobilenet", "single_resnet18", "single_effnet_b0"]
TARGET_FAR = 0.05

# Per-class image counts in ImageFolder order (sorted class names), as measured by notebook 00 (preflight).
# Used to map each cached sample index back to its class without mounting the datasets.
COUNTS = {
 "Malimg": {"Adialer.C":122,"Agent.FYI":116,"Allaple.A":2948,"Allaple.L":1591,"Alueron.gen!J":198,"Autorun.K":106,
            "C2LOP.P":146,"C2LOP.gen!g":200,"Dialplatform.B":177,"Dontovo.A":162,"Fakerean":381,"Instantaccess":431,
            "Lolyda.AA1":213,"Lolyda.AA2":184,"Lolyda.AA3":123,"Lolyda.AT":159,"Malex.gen!J":136,"Obfuscator.AD":142,
            "Rbot!gen":158,"Skintrim.N":80,"Swizzor.gen!E":128,"Swizzor.gen!I":132,"VB.AT":408,"Wintrim.BX":97,"Yuner.A":800},
 "MaleVis": {"Adposhel":494,"Agent":470,"Allaple":478,"Amonetize":497,"Androm":500,"Autorun":496,"BrowseFox":493,
             "Dinwod":499,"Elex":500,"Expiro":501,"Fasong":500,"HackKMS":499,"Hlux":500,"Injector":495,"InstallCore":500,
             "MultiPlug":499,"Neoreklami":500,"Neshta":497,"Other":1832,"Regrun":485,"Sality":499,"Snarasite":500,
             "Stantinko":500,"VBA":500,"VBKrypt":496,"Vilsel":496},
 "Dumpware10": {"Adposhel":457,"Allaple":437,"Amonetize":436,"AutoRun":196,"BrowseFox":190,"Dinwod":127,
                "InstallCore":467,"MultiPlug":488,"Other":608,"VBA":499,"Vilsel":389},
}

# Attached notebook outputs (from the Input panel). Searched first; a recursive search is the fallback.
INPUT_ROOTS = {
    "Malimg":     "/kaggle/input/notebooks/rauf41/01-auditx-malimg",
    "MaleVis":    "/kaggle/input/notebooks/rauf41/02-auditx-malevis-2",
    "Dumpware10": "/kaggle/input/notebooks/rauf41/03-auditx-dumpware10-final",
}
for ds, root in INPUT_ROOTS.items():
    print(f"{ds:<11} {root}  exists={os.path.isdir(root)}")
    if os.path.isdir(root):
        print("            contents:", sorted(os.listdir(root))[:10])

def find_run_dir(ds):
    root = INPUT_ROOTS.get(ds, "")
    for cand in [os.path.join(root, f"auditx_v2_{ds.lower()}", ds)] + \
                glob.glob(os.path.join(root, "**", f"auditx_v2_{ds.lower()}", ds), recursive=True):
        if os.path.isdir(cand) and "QUICKTEST" not in cand and glob.glob(os.path.join(cand, "seed*", "split.json")):
            return cand
    hits = [p for p in glob.glob(f"/kaggle/input/**/auditx_v2_{ds.lower()}/{ds}", recursive=True)
            if "QUICKTEST" not in p and glob.glob(os.path.join(p, "seed*", "split.json"))]
    return sorted(hits)[0] if hits else None

def class_of_index(ds):
    names = sorted(COUNTS[ds])                     # ImageFolder sorts class folder names
    bounds = np.cumsum([0] + [COUNTS[ds][n] for n in names])
    return names, bounds

RUNS = {}
for ds in DATASETS:
    d = find_run_dir(ds)
    if d is None:
        print(f"!! {ds}: no cached run found - attach the committed output of its notebook"); continue
    seeds = sorted(int(re.findall(r"seed(\d+)", p)[0]) for p in glob.glob(os.path.join(d, "seed*")))
    names, bounds = class_of_index(ds)
    ok = True
    for s in seeds:
        sp = json.load(open(os.path.join(d, f"seed{s}", "split.json")))
        unk = np.array(sp["unk"]); cls = np.searchsorted(bounds, unk, side="right") - 1
        got = {names[c]: int((cls == c).sum()) for c in np.unique(cls)}
        want = {n: COUNTS[ds][n] for n in sp["unknown_names"]}
        n_drop = sp.get("n_dropped_conflict", 0)
        # protocol v3 removes label-conflict duplicate groups, so an unknown family may have a few images fewer
        counts_ok = set(got) <= set(want) and all(0 < got.get(n, 0) <= want[n] for n in want) and sum(want.values()) - sum(got.values()) <= n_drop
        if sp.get("protocol") != REQUIRED_PROTOCOL:
            ok = False; print(f"!! {ds} seed {s}: split protocol is {sp.get('protocol', 'random-v2 (old)')}, expected {REQUIRED_PROTOCOL} - attach the NEW committed versions")
        if not counts_ok or sorted(sp["known_names"] + sp["unknown_names"] + sp.get("excluded_classes", [])) != names:
            ok = False; print(f"!! {ds} seed {s}: index->class mapping check FAILED\n   got {got}\n   want {want}")
    have = {m: all(os.path.exists(os.path.join(d, f"seed{s}", f"{m}.npz")) for s in seeds) for m in MODELS}
    print(f"{ds}: {d}\n   seeds {seeds}; mapping check {'OK' if ok else 'FAILED'}; models cached: {have}")
    if ok and all(have.values()):
        RUNS[ds] = dict(dir=d, seeds=seeds, names=names, bounds=bounds)
assert RUNS, "no usable runs - attach the outputs of notebooks 01-03"

In [ ]:
# Cell 3 - helpers
PARTS = ("train", "calib", "test", "unk")

def load(run_dir, seed, model):
    d = np.load(os.path.join(run_dir, f"seed{seed}", f"{model}.npz"), allow_pickle=True)
    return {p: {k: d[f"{p}__{k}"] for k in ("logits", "expert_logits", "weights", "feats", "y")} for p in PARTS}

def deep_ensemble(singles):
    de = {}
    for p in PARTS:
        de[p] = dict(expert_logits=np.concatenate([s[p]["expert_logits"] for s in singles], 1),
                     feats=np.concatenate([s[p]["feats"] for s in singles], 1), y=singles[0][p]["y"])
        de[p]["logits"] = de[p]["expert_logits"].mean(1)
        de[p]["weights"] = np.full((len(de[p]["y"]), 3), 1 / 3)
    return de

def scores_for(o, is_de, seed):
    sc = {p: ax.base_scores(o[p], is_de) for p in PARTS}
    knn = ax.KNNScorer(o["train"]["feats"], ax.CFG["KNN_K"], ax.CFG["KNN_MAX_TRAIN"], seed)  # same settings as the main runs
    for p in ("calib", "test", "unk"):
        sc[p]["knn"] = knn(o[p]["feats"])
    return sc

def unit(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True)

def class_centroids(feats, labels, classes):
    f = unit(feats)
    return np.stack([unit(f[labels == c].mean(0, keepdims=True))[0] for c in classes])

def prefix(n):
    return re.split(r"[.!]", n)[0].lower()

def calibrate_union(ca, cb, budget):
    """Per-axis alpha so that the union of the two alarms flags `budget` of calibration knowns."""
    lo, hi = 0.0, budget
    for _ in range(40):
        a = (lo + hi) / 2
        ta, tb = np.quantile(ca, 1 - a), np.quantile(cb, 1 - a)
        rate = ((ca > ta) | (cb > tb)).mean()
        lo, hi = (a, hi) if rate < budget else (lo, a)
    a = lo
    return np.quantile(ca, 1 - a), np.quantile(cb, 1 - a), a

In [ ]:
# Cell 5 - locate the training-seed replicates (13a-c) and score every (dataset, split, training seed, model)
from scipy.stats import rankdata, beta
def find_seed_dir(ds):
    hits = [p for p in glob.glob(f"/kaggle/input/**/auditx_v2_seeds_{ds.lower()}/{ds}", recursive=True) if "QUICKTEST" not in p]
    return sorted(hits)[0] if hits else None
SEED_DIRS = {ds: find_seed_dir(ds) for ds in RUNS}
print("seed-replicate folders:", SEED_DIRS)

def maha_scorer(train_f, train_y, n_cls, max_dim=512):
    from sklearn.covariance import LedoitWolf
    from sklearn.decomposition import PCA
    pca = PCA(n_components=min(max_dim, train_f.shape[1], len(train_f) - 1), random_state=0).fit(train_f)
    z = pca.transform(train_f); mu = np.stack([z[train_y == c].mean(0) for c in range(n_cls)])
    cov = LedoitWolf().fit(z - mu[train_y]).covariance_; ev, V = np.linalg.eigh(cov); W = V / np.sqrt(np.clip(ev, 1e-12, None)); muw = mu @ W
    def f(x):
        zw = pca.transform(x) @ W
        return ((zw ** 2).sum(1)[:, None] - 2 * zw @ muw.T + (muw ** 2).sum(1)[None, :]).min(1)
    return f

def load_any(path):
    d = np.load(path, allow_pickle=True)
    return {p: {k: d[f"{p}__{k}"] for k in ("logits", "expert_logits", "weights", "feats", "y")} for p in ("train", "calib", "test", "unk")}, \
           (json.loads(str(d["meta"])) if "meta" in d.files else None)

SC, META = {}, {}
for ds, R in RUNS.items():
    for seed in R["seeds"]:
        sp = json.load(open(os.path.join(R["dir"], f"seed{seed}", "split.json")))
        known = sp["known_names"]
        unk_fam = np.array([R["names"][c] for c in np.searchsorted(R["bounds"], np.array(sp["unk"]), side="right") - 1])
        for t in (0, 1, 2):
            base_dir = os.path.join(R["dir"], f"seed{seed}") if t == 0 else \
                       (os.path.join(SEED_DIRS[ds], f"seed{seed}", f"train{t}") if SEED_DIRS.get(ds) else None)
            if base_dir is None or not all(os.path.exists(os.path.join(base_dir, f"{m}.npz")) for m in MODELS): continue
            outs = {}
            for m in MODELS:
                outs[m], meta = load_any(os.path.join(base_dir, f"{m}.npz"))
                if meta: META[(ds, seed, m)] = meta
            outs["deep_ensemble"] = deep_ensemble([outs[s] for s in SINGLES])
            for m, o in outs.items():
                is_de = m == "deep_ensemble"
                sc = {p: ax.base_scores(o[p], is_de) for p in ("calib", "test", "unk")}
                for k in KNN_KS:
                    kn = ax.KNNScorer(o["train"]["feats"], k, ax.CFG["KNN_MAX_TRAIN"], seed)
                    for p in sc: sc[p][f"knn{k}"] = kn(o[p]["feats"])
                tf, ty = o["train"]["feats"], o["train"]["y"]
                if len(tf) > 6000:
                    ii = np.random.RandomState(seed).choice(len(tf), 6000, replace=False); tf, ty = tf[ii], ty[ii]
                mh = maha_scorer(tf, ty, len(known))
                for p in sc: sc[p]["mahalanobis"] = mh(o[p]["feats"])
                def acc(p):
                    pr = softmax(o[p]["expert_logits"], -1).mean(1) if is_de else softmax(o[p]["logits"], -1)
                    return float((pr.argmax(1) == o[p]["y"]).mean())
                SC[(ds, seed, t, m)] = dict(sc={p: {s: sc[p][s] for s in SCORES if s in sc[p]} for p in sc}, unk_fam=unk_fam,
                                            acc_calib=acc("calib"), acc_test=acc("test"))
            del outs
        print(f"{ds} split {seed}: training seeds found {[t for t in (0, 1, 2) if (ds, seed, t, 'agef_sharp') in SC]}", flush=True)
CONFIGS = sorted({(m, s) for (_, _, _, m), d in SC.items() for s in d["sc"]["test"]})
OUT = "/kaggle/working/auditx_v4_review2"; os.makedirs(OUT, exist_ok=True)
print(len(SC), "scored runs;", len(CONFIGS), "configurations")

In [ ]:
# Cell 6 - S1: selection without unknown families
def auroc(k, u):
    r = rankdata(np.r_[k, u]); return (r[len(k):].sum() - len(u) * (len(u) + 1) / 2) / (len(u) * len(k))
def zdr_at(d, s, a=0.05):
    thr = np.quantile(d["sc"]["calib"][s], 1 - a); return float((d["sc"]["unk"][s] > thr).mean())
rows = []
for (ds, seed, t, m), d in SC.items():
    for s in d["sc"]["test"]:
        rows.append(dict(dataset=ds, split=seed, train_seed=t, model=m, score=s, auroc=auroc(d["sc"]["test"][s], d["sc"]["unk"][s]),
                         zdr=zdr_at(d, s), acc_calib=d["acc_calib"]))
P = pd.DataFrame(rows); P.to_csv(f"{OUT}/S1_pool_raw.csv", index=False)
P0 = P[P.train_seed == 0]
res, scal = [], []
rng = np.random.RandomState(0)
for ds, g in P0.groupby("dataset"):
    piv = g.pivot_table(index=["model", "score"], columns="split", values="zdr")
    pool_mean = piv.values.mean(); s3 = piv.max(0).mean(); best = piv.mean(1).idxmax(); s4 = piv.loc[best].max()
    # known-data-only rule: per split, the model with the highest calibration accuracy, with the score fixed in advance (energy)
    ko = []
    for sp_, gs in g.groupby("split"):
        mbest = gs.drop_duplicates("model").set_index("model").acc_calib.idxmax()
        ko.append(gs[(gs.model == mbest) & (gs.score == "energy")].zdr.iloc[0])
    res.append(dict(dataset=ds, pool_size=len(piv), pool_mean_zdr=pool_mean, s3_test_selected=s3, s4_best_config_best_split=s4,
                    infl_s3_vs_pool_pp=100 * (s3 - pool_mean), infl_s4_vs_pool_pp=100 * (s4 - pool_mean),
                    known_only_selection_zdr=np.mean(ko), infl_s3_vs_known_only_pp=100 * (s3 - np.mean(ko))))
    for msize in [1, 2, 4, 8, 16, 32, len(piv)]:
        if msize > len(piv): continue
        vals = []
        for _ in range(300):
            sub = piv.iloc[rng.choice(len(piv), msize, replace=False)]
            vals.append(sub.max(0).mean() - sub.values.mean())
        scal.append(dict(dataset=ds, pool_size=msize, s3_inflation_pp=100 * np.mean(vals), sd=100 * np.std(vals)))
S1 = pd.DataFrame(res); S1s = pd.DataFrame(scal)
S1.to_csv(f"{OUT}/S1_selection.csv", index=False); S1s.to_csv(f"{OUT}/S1_pool_size_scaling.csv", index=False)
display(S1.round(3)); display(S1s.pivot_table(index="pool_size", columns="dataset", values="s3_inflation_pp").round(1))

In [ ]:
# Cell 7 - S2: family-level cluster bootstrap (resample unknown families and known test samples within dataset; all splits x seeds), Holm
COMPARE = [  # (configuration, reference) - two-sided
    (("deep_ensemble", "knn50"), ("agef_sharp", "entropy")), (("deep_ensemble", "energy"), ("agef_sharp", "entropy")),
    (("avg_joint", "entropy"), ("agef_sharp", "entropy")), (("single_mobilenet", "entropy"), ("agef_sharp", "entropy")),
    (("deep_ensemble", "energy"), ("single_mobilenet", "energy")), (("deep_ensemble", "knn50"), ("single_mobilenet", "knn50")),
    (("deep_ensemble", "mahalanobis"), ("single_effnet_b0", "mahalanobis")), (("deep_ensemble", "energy"), ("single_effnet_b0", "energy"))]
def boot_diff(ds, a, b, n_boot):
    keys = sorted({(s, t) for (d_, s, t, m) in SC if d_ == ds and m == a[0] and (d_, s, t, b[0]) in SC})
    fams = sorted({f for (s, t) in keys for f in SC[(ds, s, t, a[0])]["unk_fam"]})
    point = np.mean([auroc(SC[(ds, s, t, a[0])]["sc"]["test"][a[1]], SC[(ds, s, t, a[0])]["sc"]["unk"][a[1]]) -
                     auroc(SC[(ds, s, t, b[0])]["sc"]["test"][b[1]], SC[(ds, s, t, b[0])]["sc"]["unk"][b[1]]) for s, t in keys])
    rng = np.random.RandomState(1); out = []
    for _ in range(n_boot):
        fb = rng.choice(fams, len(fams), replace=True); diffs = []
        for s, t in keys:
            da, db = SC[(ds, s, t, a[0])], SC[(ds, s, t, b[0])]; uf = da["unk_fam"]
            idx = np.concatenate([np.where(uf == f)[0] for f in fb if (uf == f).any()] or [np.array([], int)])
            if len(idx) == 0: continue
            kb = rng.randint(0, len(da["sc"]["test"][a[1]]), len(da["sc"]["test"][a[1]]))
            diffs.append(auroc(da["sc"]["test"][a[1]][kb], da["sc"]["unk"][a[1]][idx]) - auroc(db["sc"]["test"][b[1]][kb], db["sc"]["unk"][b[1]][idx]))
        if diffs: out.append(np.mean(diffs))
    return point, np.array(out), len(keys), len(fams)
rows = []
for ds in RUNS:
    for a, b in COMPARE:
        pt, bs, nk, nf = boot_diff(ds, a, b, N_BOOT)
        p2 = min(1.0, 2 * min((bs <= 0).mean(), (bs >= 0).mean()))
        rows.append(dict(dataset=ds, config=f"{a[0]}+{a[1]}", reference=f"{b[0]}+{b[1]}", n_runs=nk, n_families=nf, delta_auroc=pt,
                         ci_lo=np.percentile(bs, 2.5), ci_hi=np.percentile(bs, 97.5), p_two_sided=p2))
    print(ds, "bootstrap done", flush=True)
S2 = pd.DataFrame(rows)
S2["p_holm"] = np.nan
for ds, g in S2.groupby("dataset"):
    order = g.p_two_sided.sort_values().index; m = len(order); run = 0
    for r, i in enumerate(order):
        run = max(run, min(1.0, (m - r) * S2.loc[i, "p_two_sided"])); S2.loc[i, "p_holm"] = run
S2["significant_holm"] = S2.p_holm < HOLM_ALPHA
S2.to_csv(f"{OUT}/S2_cluster_bootstrap.csv", index=False); display(S2.round(4))

In [ ]:
# Cell 8 - S3: low-FAR operating points with finite-sample thresholds, intervals and repeated calibration draws
def conformal_thr(x, a):
    n = len(x); k = int(np.ceil((n + 1) * (1 - a))); return np.inf if k > n else np.sort(x)[k - 1]
def cp_interval(k, n, conf=0.95):
    lo = beta.ppf((1 - conf) / 2, k, n - k + 1) if k > 0 else 0.0; hi = beta.ppf(1 - (1 - conf) / 2, k + 1, n - k) if k < n else 1.0
    return lo, hi
rows = []
for (ds, seed, t, m), d in SC.items():
    if t != 0: continue
    for s in ["entropy", "energy", "knn50", "mahalanobis"]:
        c, te, u = d["sc"]["calib"][s], d["sc"]["test"][s], d["sc"]["unk"][s]
        thr = conformal_thr(c, FAR_LOW); kf = int((te > thr).sum())
        lo, hi = cp_interval(kf, len(te))
        rng = np.random.RandomState(seed); pool = np.r_[c, te]; fr, zr = [], []
        for _ in range(N_REDRAWS):
            pr = rng.permutation(len(pool)); h = len(pool) // 2
            th = conformal_thr(pool[pr[:h]], FAR_LOW); fr.append((pool[pr[h:]] > th).mean()); zr.append((u > th).mean())
        rows.append(dict(dataset=ds, split=seed, model=m, score=s, n_calib=len(c), expected_exceedances=len(c) * FAR_LOW,
                         reliable=len(c) * FAR_LOW >= MIN_EXCEEDANCES, zdr=float((u > thr).mean()), far=kf / len(te), far_ci_lo=lo, far_ci_hi=hi,
                         redraw_far_mean=np.mean(fr), redraw_far_sd=np.std(fr), redraw_zdr_mean=np.mean(zr), redraw_zdr_sd=np.std(zr)))
S3 = pd.DataFrame(rows); S3.to_csv(f"{OUT}/S3_lowfar_raw.csv", index=False)
S3s = S3.groupby(["dataset", "model", "score"]).agg(n_calib=("n_calib", "mean"), reliable=("reliable", "mean"), zdr=("zdr", "mean"), far=("far", "mean"),
      far_ci_lo=("far_ci_lo", "mean"), far_ci_hi=("far_ci_hi", "mean"), redraw_far_sd=("redraw_far_sd", "mean"),
      redraw_zdr_mean=("redraw_zdr_mean", "mean"), redraw_zdr_sd=("redraw_zdr_sd", "mean")).reset_index()
S3s.to_csv(f"{OUT}/S3_lowfar_summary.csv", index=False)
display(S3s[S3s.model.isin(["deep_ensemble", "agef_sharp", "single_mobilenet"])].round(4))

In [ ]:
# Cell 9 - S4: rankings at KNN k = 5, 10, 50 (AUROC, mean over splits and training seeds)
K4 = P[P.score.str.startswith("knn")].pivot_table(index=["dataset", "model"], columns="score", values="auroc").round(4)
K4.to_csv(f"{OUT}/S4_knn_k.csv"); display(K4)
for ds in RUNS:
    for k in KNN_KS:
        g = P[(P.dataset == ds) & (P.score == f"knn{k}")].groupby("model").auroc.mean().sort_values(ascending=False)
        print(f"{ds:<11} k={k:<3} ranking: " + " > ".join(f"{m}({v:.3f})" for m, v in g.items()))

In [ ]:
# Cell 10 - S5: AGEF-Net per-seed AUROC and training curves (seed 0 histories)
import matplotlib.pyplot as plt
ps = P[P.model.isin(["agef_sharp", "avg_joint", "deep_ensemble", "single_mobilenet"]) & P.score.isin(["entropy", "energy", "knn50"])]
PS = ps.pivot_table(index=["dataset", "model", "score", "split"], columns="train_seed", values="auroc").round(3)
PS.to_csv(f"{OUT}/S5_per_seed_auroc.csv"); display(PS.loc[(slice(None), "agef_sharp", "entropy"), :])
fig, axs = plt.subplots(1, len(RUNS), figsize=(5 * len(RUNS), 3.2), squeeze=False)
curve_rows = []
for a_, ds in zip(axs[0], RUNS):
    for (d_, seed, m), meta in META.items():
        if d_ != ds or m not in ("agef_sharp", "avg_joint", "single_mobilenet"): continue
        h = pd.DataFrame(meta.get("history", []))
        if h.empty: continue
        a_.plot(h.epoch, h.val_acc, color={"agef_sharp": "#c44e52", "avg_joint": "#4c72b0", "single_mobilenet": "#55a868"}[m], alpha=.6, lw=1,
                label=m if seed == RUNS[ds]["seeds"][0] else None)
        curve_rows.append(dict(dataset=ds, split=seed, model=m, best_epoch=int(h.val_acc.idxmax()), best_val_acc=h.val_acc.max(),
                               final_train_loss=h.train_loss.iloc[-1], val_acc_range_last5=h.val_acc.tail(5).max() - h.val_acc.tail(5).min()))
    a_.set_title(ds); a_.set_xlabel("epoch"); a_.set_ylabel("validation accuracy (%)"); a_.legend(fontsize=7); a_.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_S5_training_curves.pdf"); plt.savefig(f"{OUT}/fig_S5_training_curves.png", dpi=200); plt.show()
CR = pd.DataFrame(curve_rows); CR.to_csv(f"{OUT}/S5_training_curves_summary.csv", index=False)
display(CR.groupby(["dataset", "model"]).mean(numeric_only=True).round(3))

In [ ]:
# Cell 11 - S6: early-exit compute for known and unknown inputs separately (from notebook 01-03 dynamic outputs)
dfiles = [f for f in glob.glob("/kaggle/input/**/auditx_v2_*/dynamic_raw.csv", recursive=True) if "QUICKTEST" not in f and "seeds" not in f]
if not dfiles: raise SystemExit("S6 skipped: no dynamic_raw.csv found in the attached 01-03 outputs")
dyn = pd.concat([pd.read_csv(f) for f in dfiles], ignore_index=True)
f_m, f_all = 0.111, 0.111 + 3.627 + 0.769
dyn["gflops_known"] = f_m + (1 - dyn.exit_rate_known) * (f_all - f_m)
dyn["gflops_unknown"] = f_m + (1 - dyn.exit_rate_unknown) * (f_all - f_m)
E6 = dyn.groupby(["dataset", "target_exit"])[["exit_rate_known", "exit_rate_unknown", "gflops_known", "gflops_unknown", "gflops_per_img", "zdr", "far_test"]].mean()
E6["saving_known_pct"] = 100 * (1 - E6.gflops_known / f_all); E6["saving_unknown_pct"] = 100 * (1 - E6.gflops_unknown / f_all)
E6["saving_mixed_pct"] = 100 * (1 - E6.gflops_per_img / f_all)
E6.to_csv(f"{OUT}/S6_early_exit_split.csv"); display(E6.round(3))
print("Note: the mixed saving averages known test and unknown samples in the proportions of the evaluation set.")
print("outputs:", sorted(os.listdir(OUT)))